In [1]:
import random

# Il tuo input (Python lo vede come una tupla di 16 stringhe)
words_input = "BASHFUL", "DOC", "GRUMPY", "HAPPY", "GIF", "PDF", "TIFF", "ZIP", "EMU", "KIWI", "OSTRICH", "PENGUIN", "BANANA", "COCONUT", "MANGO", "PINEAPPLE"

def shuffle_and_stringify(words_sequence):
    # 1. Convertiamo in lista (perché le tuple non si possono modificare/shufflare)
    words_list = list(words_sequence)
    
    # 2. Mischiamo
    random.shuffle(words_list)
    
    # 3. Uniamo in una stringa unica separata da virgole per il prompt
    return ", ".join(words_list)

# Utilizzo
shuffled_string = shuffle_and_stringify(words_input)

print(f"Input per il modello:\n{shuffled_string}")

Input per il modello:
BANANA, MANGO, TIFF, COCONUT, HAPPY, BASHFUL, OSTRICH, EMU, ZIP, PENGUIN, KIWI, DOC, GRUMPY, PINEAPPLE, PDF, GIF


In [2]:
system_instruction = """
You are an expert solver of the NYT Connections game.
Your task is to receive 16 words and group them into 4 precise semantic groups of 4 words each.
BE CAREFUL: some words have double senses!

STRICT RULES:
1. Analyze the relationships between the words step by step in your thinking process.
2. The final output (after thinking) must be a valid JSON object ONLY.
3. Do not add greetings, explanations, or other text outside the JSON.
4. The format must be:
[
{"category": "CATEGORY_NAME", "words": ["p1", "p2", "p3", "p4"]},
...
]
"""

messages = [
    {"role": "system", "content": system_instruction}, # La variabile definita sopra
    {"role": "user", "content": f"Solve this puzzle. Words are: {shuffled_string}"}
]

In [3]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# --- CONFIGURAZIONE PER RTX 3070 (4-bit) ---
# Questo riduce il modello per farlo entrare negli 8GB di VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Caricamento modello in corso (4-bit)...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"  # Gestisce automaticamente GPU
)

# --- LOGICA CONNECTIONS ---

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

print("Generazione risposta...")
outputs = model.generate(
    **inputs, 
    max_new_tokens=1500, # Aumentato perché il "pensiero" consuma token
    do_sample=False,      # Deterministico (temperatura 0)
    pad_token_id=tokenizer.eos_token_id
)

full_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# Parsing pulito
if "</think>" in full_response:
    final_answer = full_response.split("</think>")[1].strip()
else:
    final_answer = full_response

print("\n--- RISPOSTA JSON ---")
print(final_answer)

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Caricamento modello in corso (4-bit)...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generazione risposta...

--- RISPOSTA JSON ---
[
{"category": "Food", "words": ["BANANA", "MANGO", "COCONUT", "PINEAPPLE"]},
{"category": "Animals", "words": ["HAPPY", "BASHFUL", "EMU", "PENGUIN"]},
{"category": "Technology", "words": ["ZIP", "PDF", "GIF", "DOC"]},
{"category": "Emotions", "words": ["GRUMPY", "OSTRICH", "HAPPY", "BASHFUL"]},
]
